In [17]:
# Load libraries

import os
import pandas
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats
import sklearn.model_selection
import shutil

# import my extended pySuStaIn model
from apoe4_sustain import ZscoreSustain_APOE4
# import simulation function
from genetics_simulation_utils import simulate_apoe_sustain_cohort #genetics_simulation,


1. Simulate biomarkers using zscore SuStaIn model and a categorical variable: APOE carrier status based on fixed genetic probabilities

In [18]:
# define hyper-parameters of simulation
# Configure dataset dimensions
M_patients = 1000
N_biomarkers = 5

In [19]:
# N_subtypes = 1
# gt_fractions = np.array([1.0])
# W_true_target = np.array([
#     [1/3, 1/3, 1/3],  # Subtype 0: High Category 0 anchor[1/3, 1/3, 1/3]   # Subtype 1: High Category 2 anchor
# ])


In [20]:
N_subtypes = 2
gt_fractions = np.array([0.45, 0.55])


#Define the explicit ground-truth genetic profiles for this sandbox run
W_true_target = np.array([
    [0.80, 0.15, 0.05],  # Subtype 0: High Category 0 anchor
    [0.10, 0.30, 0.60]   # Subtype 1: High Category 2 anchor
])

# # #Uniform case
# # W_true_target = np.array([
# #     [1/3, 1/3, 1/3],  # Subtype 0: High Category 0 anchor
# #     [1/3, 1/3, 1/3]   # Subtype 1: High Category 2 anchor
# # ])


In [21]:
N_subtypes = 3
W_true_target = np.array([
    [0.80, 0.15, 0.05],  # Subtype 0: High Category 0 anchor
    [0.10, 0.30, 0.60],
    [0.40, 0.40,0.20] ,  # Subtype 1: High Category 2 anchor
])
gt_fractions = np.array([0.60, 0.30, 0.10])

In [22]:

# Generate the dataset in-memory
df, Z_vals, Z_max, gt_ordering, gt_fractions, W_true = simulate_apoe_sustain_cohort(N = N_biomarkers,              
                                                                            M = M_patients,            # number of observations (e.g. subjects)
                                                                            N_S_gt = N_subtypes,         # number of ground truth subtypes
                                                                            gt_f = gt_fractions,         # could also be None
                                                                            W_true = W_true_target,      # genetic_weights
                                                                            genetic_signal_strength = None # could be 'uniform', 'moderate' or 'strong
                                                                            )
df

Generator(PCG64)


,Biomarker 0,Biomarker 1,Biomarker 2,Biomarker 3,Biomarker 4,gt_subtypes,gt_stages,apoe_status
0,1.920169,-0.032609,0.272496,1.270027,-0.192990,0,0.0,2
1,0.899629,-0.203208,3.012573,-0.634990,0.726564,0,0.0,0
2,0.663162,-0.047887,0.713764,0.062976,-0.547454,0,0.0,0
3,0.882771,-0.756041,-2.278049,1.065845,0.229569,0,0.0,0
4,-0.736323,0.747099,-0.388613,0.621893,-0.998311,0,0.0,0
...,...,...,...,...,...,...,...,...
995,3.190865,6.237039,6.436933,6.503458,3.041084,2,16.0,2
996,4.166372,0.275068,-0.750912,3.667890,0.739913,2,4.0,1
997,5.420762,3.905662,3.717899,4.396853,2.353304,2,14.0,2
998,1.868744,1.831696,0.563172,1.507358,3.088833,3,4.0,1


In [ ]:
# # Generate the dataset in-memory
# old function I no longer use
# df, Z_vals, Z_max, gt_ordering, W_true = genetics_simulation(
#     N=N_biomarkers, 
#     M=M_patients, 
#     N_S_gt=N_subtypes, 
#     gt_fractions=gt_fractions, 
#     W_true=W_true_target, 
#     use_midpoints=False, # Set to True if you want to test your supervisor's midpoint variant
#     save=False
# )

# df

In [24]:
# Extract raw arrays for the model execution
BiomarkerNames = [f'Biomarker {i}' for i in range(N_biomarkers)]
X_data = df[BiomarkerNames].values
y_genetics = df['apoe_status'].values

print(f"Generated data matrix shape: {X_data.shape}")
print(f"Generated genetic vector shape: {y_genetics.shape}")

Generated data matrix shape: (1000, 5)
Generated genetic vector shape: (1000,)


In [25]:
N_startpoints = 15
N_S_max = N_subtypes
N_iterations_MCMC = int(1e4)
#output_folder = os.path.join(os.getcwd(), 'MRI_Sustain_5features_all')



In [26]:
dataset_name = 'simulated_data_genetics_model'
output_folder = os.path.join(os.getcwd(), dataset_name)
sustain_input = ZscoreSustain_APOE4(X_data,
                                Z_vals,
                                Z_max,
                                BiomarkerNames,
                                N_startpoints,
                                N_S_max, 
                                N_iterations_MCMC, 
                                output_folder, 
                                dataset_name, 
                                False,
                                apoe4_status = y_genetics,
                                #apoe4_status = gender_status_array,
                                apoe_flag = True)

1000 patients initially
 -> Detected 3 unique genetic categories: [0 1 2]
1000 patients with non null apoe4 carrier status
 -> [Genetics Setup] Global cohort background frequencies calculated: [0.549 0.223 0.228]
Running genetic weighted SuStaIn 


Run SuStaIn 

In [27]:
# delete previous pickle files
if os.path.exists(output_folder):
    shutil.rmtree(output_folder)

In [28]:
# make the output directory if it's not already created
if not os.path.isdir(output_folder):
    os.mkdir(output_folder)

In [ ]:

samples_sequence,     \
ml_f_EM,            \
ml_subtype,         \
prob_ml_subtype,    \
ml_stage,           \
prob_ml_stage,      \
prob_subtype_stage,         = sustain_input.run_sustain_algorithm()


Failed to find pickle file: /Users/mihaelacroitor/apoe4_informed/simulated_data_genetics_model/pickle_files/simulated_data_genetics_model_subtype0.pickle. Running SuStaIn model for 0 subtype.
Finding ML solution to 1 cluster problem
Using alternating method
Genetic weight estimation converged at iteration 1
EM converged in 3 iterations
Using alternating method
Genetic weight estimation converged at iteration 1
EM converged in 3 iterations
Using alternating method
Genetic weight estimation converged at iteration 1
EM converged in 3 iterations
Using alternating method
Genetic weight estimation converged at iteration 1
EM converged in 3 iterations
Using alternating method
Genetic weight estimation converged at iteration 1
EM converged in 3 iterations
Using alternating method
Genetic weight estimation converged at iteration 1
EM converged in 3 iterations
Using alternating method
Genetic weight estimation converged at iteration 1
EM converged in 3 iterations
Using alternating method
Genetic

MCMC Iteration:  53%|█████▎    | 5272/10000 [00:02<00:02, 2010.80it/s]

### Proof 1 Does the model recover correct params?

## Does the EM converge to max likelihood?

In [ ]:
s = N_subtypes - 1
pickle_filename_s = output_folder + '/pickle_files/' + dataset_name + '_subtype' + str(s) + '.pickle'
pk = pandas.read_pickle(pickle_filename_s)
ml_likelihood_mat_EM = pk["ml_likelihood_mat_EM"]

ml_genetic_weights_EM = pk["ml_genetic_weights_EM"]
ml_sequence_EM = pk["ml_sequence_EM"]
em_likelihood_histories = pk["em_likelihood_histories"]

In [ ]:
em_likelihood_histories[:,2]

In [ ]:
print(f"Matrix Shape: {em_likelihood_histories.shape}") # Should be (100, 15)
print(f"Total Non-NaN points per startpoint column:")
print(np.sum(~np.isnan(em_likelihood_histories), axis=0))

In [ ]:
import numpy as np
import pandas as pd

# 1. Load your saved model pickle
s = N_subtypes - 1 # Subtype loop index indicator
pickle_filename_s = f"{output_folder}/pickle_files/{dataset_name}_subtype{s}.pickle"
pk = pd.read_pickle(pickle_filename_s)

ml_likelihood_mat_EM = pk["ml_likelihood_mat_EM"]
em_likelihood_histories = pk["em_likelihood_histories"]
print('EM lik history',em_likelihood_histories.shape)

# 2. Identify the global maximum value found in this run
global_max_val = np.max(ml_likelihood_mat_EM)

# 3. Find which startpoint IDs reached this max (allowing a tiny floating-point tolerance)
epsilon = 1e-2
winning_startpoints_mask = np.abs(ml_likelihood_mat_EM - global_max_val) <= epsilon
winning_indices = np.where(winning_startpoints_mask)[0]
print(winning_indices)

# 4. Calculate convergence steps across all runs by counting non-NaN markers
all_convergence_steps = np.sum(~np.isnan(em_likelihood_histories), axis=0)

# 5. Filter to keep ONLY the winning startpoints
winning_speeds = all_convergence_steps[winning_indices]

# ---------------------------------------------------------------------
# PRESENTATION REPORT
# ---------------------------------------------------------------------
print("=" * 60)
print(f" WINNING STARTPOINTS CONVERGENCE SPEED REPORT (Max Lik: {global_max_val:.2f})")
print("=" * 60)
print(f" Total Startpoints: {len(ml_likelihood_mat_EM)}")
print(f" Global Winners:    {len(winning_indices)} / {len(ml_likelihood_mat_EM)}")
print("-" * 60)

for idx, steps in zip(winning_indices, winning_speeds):
    print(f" -> Startpoint {idx + 1:2d}: Hit True Global Max in {steps:2d} iterations")

print("-" * 60)
print(f" Avg speed of winning tracks: {np.mean(winning_speeds):.1f} iterations")
print("=" * 60)

In [ ]:
# compare estimated vs true params
print("True seq", gt_ordering)
print("Estimated seq", ml_sequence_EM)
print("Estimated fractions",ml_f_EM)
print("True genetic weights",W_true_target)
print("Estimated genetics",ml_genetic_weights_EM)

In [ ]:
# calculate actual max likelihood from simulated parameters
sustainData = sustain_input._AbstractSustain__sustainData
true_max_lik,_ ,_, _, _ = sustain_input._calculate_likelihood(sustainData, gt_ordering,gt_fractions, W_true)
print('True max',true_max_lik)

# did any starting points converge?
epsilon_tolerance = 0.1
converged_to_true_lik = (ml_likelihood_mat_EM >= true_max_lik) | \
                       (np.abs(ml_likelihood_mat_EM - true_max_lik) <= epsilon_tolerance)
gt_indices = np.where(converged_to_true_lik)[0]
converged = len(gt_indices)
print(f" {converged} starting points converged to true max lik {true_max_lik}")

# find max EM lik, and how many times it is reached;
max_val = np.max(ml_likelihood_mat_EM)
max_indices = np.where(np.isclose(ml_likelihood_mat_EM, max_val))[0]
occurrence_count = len(max_indices)
print(f"Maximum Likelihood Value: {max_val} reached by {occurrence_count} startingpoints")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Set up the figure size
plt.figure(figsize=(10, 6))

# 2. Plot the array directly
# Matplotlib plots each column (startpoint) as a separate colored line
plt.plot(em_likelihood_histories, linewidth=1.5, alpha=0.8)

# 3. Add your ground-truth maximum likelihood as a horizontal dotted line
# 'ls=":"' makes it dotted, 'color="black"' ensures high contrast, 'zorder=5' keeps it on top
plt.axhline(y=true_max_lik, color="black", linestyle=":", linewidth=2, label=f"True Max Likelihood ({true_max_lik:.2f})")


# 4. Add styling and descriptive labels
plt.xlabel("EM Iteration Step", fontsize=12)
plt.ylabel("Log-Likelihood", fontsize=12)
plt.title(f"EM Convergence History Across {em_likelihood_histories.shape[1]} Starting Points", fontsize=14)

plt.legend()
# 4. Clean up the grid presentation
plt.grid(True, linestyle=":", alpha=0.6)

# 5. Show the plot
plt.show()

In [ ]:
# for each subtype model
for s in range(N_S_max):
    # load pickle file (SuStaIn output) and get the sample log likelihood values
    pickle_filename_s = output_folder + '/pickle_files/' + dataset_name + '_subtype' + str(s) + '.pickle'
    pk = pandas.read_pickle(pickle_filename_s)
    samples_likelihood = pk["samples_likelihood"]
    
    # plot the values as a line plot
    fig0 = plt.figure(0)
    plt.plot(range(N_iterations_MCMC), samples_likelihood, label="model" + str(s))
    plt.legend(loc='upper right')
    plt.xlabel('MCMC samples')
    plt.ylabel('Log likelihood')
    plt.title('MCMC trace')
    
    # plot the values as a histogramp plot
    fig1 = plt.figure(1)
    plt.hist(samples_likelihood, label="model" + str(s))
    plt.legend(loc='upper right')
    plt.xlabel('Log likelihood')  
    plt.ylabel('Number of samples')  
    plt.title('Histograms of model likelihood')
fig0.savefig(output_folder+'/mcmc_trace.png')
fig1.savefig(output_folder+'/mcmc_histogram.png')


In [ ]:

use_outlier = False
for s in range(N_S_max):
    print(s)
    #s = 1 # 1 split = 2 subtypes
    M = len(X_data) 

    # get the sample sequences and f
    pickle_filename_s = output_folder + '/pickle_files/' + dataset_name + '_subtype' + str(s) + '.pickle'
    pk = pandas.read_pickle(pickle_filename_s)
    samples_sequence = pk["samples_sequence"]
    samples_f = pk["samples_f"]
    ml_f_EM = pk["ml_f_EM"]
    print(ml_f_EM)
    if use_outlier:
        subtype_order = np.argsort(ml_f_EM[:-1])
    else:
        subtype_order = np.argsort(ml_f_EM)
    print(subtype_order)
    # use this information to plot the positional variance diagrams
    tmp=ZscoreSustain_APOE4._plot_sustain_model(sustain_input,samples_sequence,samples_f,M,subtype_order=subtype_order,biomarker_labels=BiomarkerNames)
    plt.savefig(output_folder + f"/pos_var_diagrams{s}.png")
_ = plt.suptitle('Figure 4: APOE-Informed SuStaIn output')

# Output a figure showing the ground truth
gt_sequence = gt_ordering
gt_f        = gt_fractions
temp_gt_sequence = np.tile(np.reshape(gt_sequence,(gt_sequence.shape[0],gt_sequence.shape[1],1)),100)
temp_gt_f = np.asarray(gt_f).reshape(len(gt_f),1)
dynamic_gt_order = np.arange(gt_sequence.shape[0])
ZscoreSustain_APOE4._plot_sustain_model(sustain_input,temp_gt_sequence,temp_gt_f,M,subtype_order=dynamic_gt_order)
_ = plt.suptitle('Figure 3: Ground truth progression pattern')


Run baseline SuStaIn to compare

In [ ]:
dataset_name = 'simulated_data_baseline_SuStaIn'
output_folder = os.path.join(os.getcwd(), dataset_name)
sustain_bl = ZscoreSustain_APOE4(X_data,
                                Z_vals,
                                Z_max,
                                BiomarkerNames,
                                N_startpoints,
                                N_subtypes, 
                                N_iterations_MCMC, 
                                output_folder, 
                                dataset_name, 
                                False,
                                apoe4_status = y_genetics,
                                #apoe4_status = gender_status_array,
                                apoe_flag = False)

In [ ]:
# delete previous pickle files
if os.path.exists(output_folder):
    shutil.rmtree(output_folder)

In [ ]:
# make the output directory if it's not already created
if not os.path.isdir(output_folder):
    os.mkdir(output_folder)

In [ ]:
ml_sequence_EM,     \
ml_f_EM,            \
ml_subtype,         \
prob_ml_subtype,    \
ml_stage,           \
prob_ml_stage,      \
prob_subtype_stage,         = sustain_bl.run_sustain_algorithm()

In [ ]:
# calculate actual max likelihood from simulated parameters

sustainData = sustain_bl._AbstractSustain__sustainData

true_max_lik,_ ,_, _, _ = sustain_bl._calculate_likelihood(sustainData, gt_ordering,gt_fractions)

print(true_max_lik)
# see how many startingpoints achieved that

# look at last model (N_S == true)
#s = N_subtypes - 1
s = 0
pickle_filename_s = output_folder + '/pickle_files/' + dataset_name + '_subtype' + str(s) + '.pickle'
print(pickle_filename_s)
pk = pandas.read_pickle(pickle_filename_s)
ml_likelihood_mat_EM = pk["ml_likelihood_mat_EM"]

# did any starting points converge?
max_indices = np.where(np.isclose(ml_likelihood_mat_EM, true_max_lik))[0]
occurrence_count = len(max_indices)
print(f" {occurrence_count} starting points converged to true max lik {true_max_lik}")
#print(f"Reached by startpoints:   {max_indices}")

# find max EM lik, and how many times it is reached;
max_val = np.max(ml_likelihood_mat_EM)
max_indices = np.where(np.isclose(ml_likelihood_mat_EM, max_val))[0]
occurrence_count = len(max_indices)
print(f"Maximum Likelihood Value: {max_val} reached by {occurrence_count} startingpoints")
#print(f"Reached by startpoints:   {max_indices}")

ml_sequence_EM = pk["ml_sequence_EM"]
ml_f_EM = pk["ml_f_EM"]
print('ML seq EM', ml_sequence_EM)
print('True simulated seq', gt_ordering)

em_likelihood_histories = pk["em_likelihood_histories"]
print('EM lik hist',em_likelihood_histories)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Set up the figure size
plt.figure(figsize=(10, 6))

# 2. Plot the array directly
# Matplotlib plots each column (startpoint) as a separate colored line
plt.plot(em_likelihood_histories, linewidth=1.5, alpha=0.8)

# 3. Add your ground-truth maximum likelihood as a horizontal dotted line
# 'ls=":"' makes it dotted, 'color="black"' ensures high contrast, 'zorder=5' keeps it on top
plt.axhline(y=true_max_lik, color="black", linestyle=":", linewidth=2, label=f"True Max Likelihood ({true_max_lik:.2f})")


# 4. Add styling and descriptive labels
plt.xlabel("EM Iteration Step", fontsize=12)
plt.ylabel("Log-Likelihood", fontsize=12)
plt.title(f"Baseline SuStain EM Convergence History Across {em_likelihood_histories.shape[1]} Starting Points", fontsize=14)

plt.legend()
# 4. Clean up the grid presentation
plt.grid(True, linestyle=":", alpha=0.6)

# 5. Show the plot
plt.show()

In [ ]:
# Extract the active data container
sustainData = sustain_bl._AbstractSustain__sustainData

# Calculate the likelihood using the EXACT sequence and weights the model chose
test_loglike, _, _, _, _ = sustain_bl._calculate_likelihood(
    sustainData, 
    ml_sequence_EM,     # Your optimized sequence
    ml_f_EM            # Your optimized fractions
    #ml_genetic_weights_EM  # Pass the OPTIMIZED weights, not W_true!
)

print(f"Likelihood using Optimized EM Weights: {test_loglike:.2f}")
print(f"Final Plateau Level on your Plot:     {np.max(em_likelihood_histories[:,0])}")

In [ ]:
# for each subtype model
for s in range(N_S_max):
    # load pickle file (SuStaIn output) and get the sample log likelihood values
    pickle_filename_s = output_folder + '/pickle_files/' + dataset_name + '_subtype' + str(s) + '.pickle'
    pk = pandas.read_pickle(pickle_filename_s)
    samples_likelihood = pk["samples_likelihood"]
    
    # plot the values as a line plot
    fig0 = plt.figure(0)
    plt.plot(range(N_iterations_MCMC), samples_likelihood, label="model" + str(s))
    plt.legend(loc='upper right')
    plt.xlabel('MCMC samples')
    plt.ylabel('Log likelihood')
    plt.title('MCMC trace')
    
    # plot the values as a histogramp plot
    fig1 = plt.figure(1)
    plt.hist(samples_likelihood, label="model" + str(s))
    plt.legend(loc='upper right')
    plt.xlabel('Log likelihood')  
    plt.ylabel('Number of samples')  
    plt.title('Histograms of model likelihood')
fig0.savefig(output_folder+'/mcmc_trace.png')
fig1.savefig(output_folder+'/mcmc_histogram.png')


In [ ]:
# Let's plot positional variance diagrams to interpret the subtype progressions
use_outlier = False
for s in range(N_S_max):
    print(s)
    #s = 1 # 1 split = 2 subtypes
    M = len(X_data) 

    # get the sample sequences and f
    pickle_filename_s = output_folder + '/pickle_files/' + dataset_name + '_subtype' + str(s) + '.pickle'
    pk = pandas.read_pickle(pickle_filename_s)
    samples_sequence = pk["samples_sequence"]
    samples_f = pk["samples_f"]
    ml_f_EM = pk["ml_f_EM"]
    print(ml_f_EM)
    if use_outlier:
        subtype_order = np.argsort(ml_f_EM[:-1])
    else:
        subtype_order = np.argsort(ml_f_EM)
    print(subtype_order)
    # use this information to plot the positional variance diagrams
    tmp=ZscoreSustain_APOE4._plot_sustain_model(sustain_input,samples_sequence,samples_f,M,subtype_order=subtype_order,biomarker_labels=BiomarkerNames)
    plt.savefig(output_folder + f"/pos_var_diagrams{s}.png")
    _ = plt.suptitle('Figure 3: Baseline SuStaIn progression pattern')

# Output a figure showing the ground truth
gt_sequence = gt_ordering
gt_f        = gt_fractions
temp_gt_sequence = np.tile(np.reshape(gt_sequence,(gt_sequence.shape[0],gt_sequence.shape[1],1)),100)
temp_gt_f = np.asarray(gt_f).reshape(len(gt_f),1)
ZscoreSustain_APOE4._plot_sustain_model(sustain_input,temp_gt_sequence,temp_gt_f,M,subtype_order=(0,1,2))
_ = plt.suptitle('Figure 3: Ground truth progression pattern')

In [ ]:
print('Estimated seq',ml_sequence_EM)
print('True seq',gt_ordering)